In [1]:
# === CELL INSTALL: ENSURE ONNXRUNTIME IS AVAILABLE (no internet needed) ===
import sys, os, glob as _glob, subprocess
try:
    import onnxruntime as _ort_test
    print(f'onnxruntime {_ort_test.__version__} already available \u2705')
    del _ort_test
except ImportError:
    _wheel_dirs = set()
    for _whl in _glob.glob('/kaggle/input/**/*.whl', recursive=True):
        if 'onnxruntime' in os.path.basename(_whl):
            _wheel_dirs.add(os.path.dirname(_whl))
    if not _wheel_dirs:
        raise RuntimeError('onnxruntime not found and no .whl found in /kaggle/input/.')
    _find_links = []
    for _d in _wheel_dirs:
        _find_links += ['--find-links', _d]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    '--no-index', *_find_links, 'onnxruntime'], check=True)
    import onnxruntime as _ort_test
    print(f'onnxruntime {_ort_test.__version__} installed from wheel \u2705')
    del _ort_test

onnxruntime 1.24.4 installed from wheel ✅


# BirdCLEF 2026 — Inference v25 (Larger PerchGRU + Mel Ensemble)
## ONNX Perch + BiGRU (hidden=768, layers=3, attn) + ResNet18(v25) + EfficientNet-B0(v25)

### Architecture
- **Perch branch (5 folds)**: ONNX `perch_v2_cpu.onnx` → PerchGRU v25 (768-hidden, 3-layer BiGRU)
- **Mel branch — ResNet18 (5 folds)**: Log-mel at 16kHz → BirdCLEFModel(resnet18) trained on correct species list
- **Mel branch — EfficientNet-B0 (5 folds)**: same
- **Ensemble**: weighted mean (gru=0.6, mel=0.4 — tune after seeing individual scores)

### Required Kaggle inputs
1. `birdclef-2026`
2. `chiragggg/birdclef-2026-perch-onnx`
3. `chiragggg/birdclef-2026-weights-v25` (output of train-v25-gru-mel)

In [2]:
# === CELL 0: IMPORTS & CONFIG ===
import os, warnings, traceback, gc
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import onnxruntime as ort
from scipy.ndimage import gaussian_filter1d

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast

import timm
from tqdm import tqdm

warnings.filterwarnings('ignore')

CFG = dict(
    folds         = 5,
    device        = 'cuda' if torch.cuda.is_available() else 'cpu',

    # Perch / GRU branch (v25 — larger)
    perch_sr      = 32000,
    perch_seconds = 5,
    perch_emb_dim = 1536,
    perch_batch   = 16,
    gru_hidden    = 768,   # upgraded from 512
    gru_layers    = 3,     # upgraded from 2

    # Mel models branch (v25 — correct species list)
    mel_sr        = 16000,
    mel_seconds   = 5,
    n_mels        = 64,
    n_fft         = 1024,
    hop_length    = 320,
    fmin          = 60,
    fmax          = 8000,

    # Post-processing
    gauss_sigma   = 1.0,

    # Ensemble weights
    # v25 mel models are trained on correct species list — expected to be competitive
    # Start at 0.6/0.4 and tune based on LB.
    # Set mel_weight=0.0 to test GRU only.
    gru_weight    = 0.9,
    mel_weight    = 0.1,
)
CFG['perch_target'] = CFG['perch_sr'] * CFG['perch_seconds']   # 160,000
CFG['mel_target']   = CFG['mel_sr']  * CFG['mel_seconds']      # 80,000

device = torch.device(CFG['device'])
torch.set_num_threads(os.cpu_count() or 4)
print(f'Device       : {device}')
print(f'onnxruntime  : {ort.__version__}')
print(f'GRU weight   : {CFG["gru_weight"]}  Mel weight: {CFG["mel_weight"]}')
print(f'GRU arch     : hidden={CFG["gru_hidden"]}, layers={CFG["gru_layers"]}')

Device       : cpu
onnxruntime  : 1.24.4
GRU weight   : 0.9  Mel weight: 0.1
GRU arch     : hidden=768, layers=3


In [3]:
# === CELL 1: PATHS & SPECIES ===
def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TAXONOMY_CSV   = _first_existing(
    '/kaggle/input/birdclef-2026/taxonomy.csv',
    '/kaggle/input/competitions/birdclef-2026/taxonomy.csv',
)
TEST_AUDIO     = _first_existing(
    '/kaggle/input/birdclef-2026/test_soundscapes',
    '/kaggle/input/competitions/birdclef-2026/test_soundscapes',
)
SAMPLE_SUB     = _first_existing(
    '/kaggle/input/birdclef-2026/sample_submission.csv',
    '/kaggle/input/competitions/birdclef-2026/sample_submission.csv',
)
CKPT_DIR = _first_existing(
    '/kaggle/input/birdclef-2026-weights-v25',
    '/kaggle/input/datasets/chiragggg/mel-gru',
)

_onnx_candidates = [
    '/kaggle/input/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/model.onnx',
]
ONNX_PATH = None
for _c in _onnx_candidates:
    if os.path.exists(_c):
        ONNX_PATH = _c
        print(f'ONNX: {ONNX_PATH}')
        break
if ONNX_PATH is None:
    import glob
    print(f'ONNX not found. Available: {glob.glob("/kaggle/input/**/*.onnx", recursive=True)}')

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

print(f'Species  : {n_classes}')
print(f'CKPT_DIR : {CKPT_DIR}')

ONNX: /kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx
Species  : 234
CKPT_DIR : /kaggle/input/datasets/chiragggg/mel-gru


In [4]:
# === CELL 2: MODEL DEFINITIONS ===

class AttentionPool(nn.Module):
    def __init__(self, d_in: int):
        super().__init__()
        self.q = nn.Linear(d_in, 1)
    def forward(self, h, mask=None):
        scores = self.q(h).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))
        w = torch.softmax(scores, dim=-1)
        return (h * w.unsqueeze(-1)).sum(1)


class PerchGRU(nn.Module):
    def __init__(self, n_classes: int, emb_dim: int = 1536,
                 hidden: int = 768, n_layers: int = 3, dropout: float = 0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 768),
            nn.GELU(),
        )
        self.gru = nn.GRU(
            input_size=768, hidden_size=hidden,
            num_layers=n_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        d_gru = hidden * 2
        self.attn = AttentionPool(d_gru)
        self.head = nn.Sequential(
            nn.LayerNorm(d_gru),
            nn.Dropout(0.2),
            nn.Linear(d_gru, n_classes),
        )

    def forward(self, x):
        single = (x.dim() == 2)
        if single: x = x.unsqueeze(1)
        z    = self.proj(x)
        h, _ = self.gru(z)
        out  = self.head(h)   # (B, T, n_classes)
        if single: out = out.squeeze(1)
        return out


class BirdCLEFModel(nn.Module):
    def __init__(self, arch: str, n_classes: int, pretrained: bool = False):
        super().__init__()
        if arch == 'resnet18':
            base    = timm.create_model('resnet18', pretrained=pretrained, in_chans=1)
            n_feats = base.fc.in_features
            base.fc = nn.Identity()
        elif arch == 'efficientnet_b0':
            base            = timm.create_model('efficientnet_b0', pretrained=pretrained, in_chans=1)
            n_feats         = base.classifier.in_features
            base.classifier = nn.Identity()
        else:
            raise ValueError(f'Unknown arch: {arch}')
        self.backbone = base
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Linear(n_feats, 512), nn.ReLU(),
            nn.Dropout(0.4), nn.Linear(512, n_classes),
        )

    def forward(self, x):
        feats = self.backbone(x)
        if feats.dim() == 4:
            feats = self.pool(feats).flatten(1)
        return self.head(feats)


print('\u2705 PerchGRU v25 + BirdCLEFModel defined')

✅ PerchGRU v25 + BirdCLEFModel defined


In [5]:
# === CELL 3: LOAD CHECKPOINTS ===
def _load_ckpts(ModelClass, ckpt_names, ckpt_dir, **model_kwargs):
    models, missing = [], []
    for name in ckpt_names:
        ckpt = Path(ckpt_dir) / name
        if not ckpt.exists():
            missing.append(str(ckpt))
            continue
        m = ModelClass(**model_kwargs).to(device)
        m.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
        m.eval()
        models.append(m)
        print(f'   \u2705 {name}')
    for p in missing:
        print(f'   \u26a0\ufe0f  MISSING: {p}')
    return models


print('Loading PerchGRU v25...')
gru_models = _load_ckpts(
    PerchGRU,
    [f'perch_gru_v25_fold{i}.pt' for i in range(CFG['folds'])],
    CKPT_DIR,
    n_classes=n_classes, emb_dim=CFG['perch_emb_dim'],
    hidden=CFG['gru_hidden'], n_layers=CFG['gru_layers'],
)

print('\nLoading ResNet18 v25...')
resnet_models = _load_ckpts(
    BirdCLEFModel,
    [f'resnet18_v25_fold{i}.pt' for i in range(CFG['folds'])],
    CKPT_DIR,
    arch='resnet18', n_classes=n_classes,
)

print('\nLoading EfficientNet-B0 v25...')
effnet_models = _load_ckpts(
    BirdCLEFModel,
    [f'efficientnet_b0_v25_fold{i}.pt' for i in range(CFG['folds'])],
    CKPT_DIR,
    arch='efficientnet_b0', n_classes=n_classes,
)

print(f'\nGRU: {len(gru_models)}  ResNet18: {len(resnet_models)}  EfficientB0: {len(effnet_models)}')

Loading PerchGRU v25...
   ✅ perch_gru_v25_fold0.pt
   ✅ perch_gru_v25_fold1.pt
   ✅ perch_gru_v25_fold2.pt
   ✅ perch_gru_v25_fold3.pt
   ✅ perch_gru_v25_fold4.pt

Loading ResNet18 v25...
   ✅ resnet18_v25_fold0.pt
   ✅ resnet18_v25_fold1.pt
   ✅ resnet18_v25_fold2.pt
   ✅ resnet18_v25_fold3.pt
   ✅ resnet18_v25_fold4.pt

Loading EfficientNet-B0 v25...
   ✅ efficientnet_b0_v25_fold0.pt
   ✅ efficientnet_b0_v25_fold1.pt
   ✅ efficientnet_b0_v25_fold2.pt
   ✅ efficientnet_b0_v25_fold3.pt
   ✅ efficientnet_b0_v25_fold4.pt

GRU: 5  ResNet18: 5  EfficientB0: 5


In [6]:
# === CELL 4: LOAD ONNX SESSION + SANITY CHECK ===
_ort_session  = None
_onnx_inp     = None
_onnx_emb_key = None
_onnx_ready   = False

if ONNX_PATH is None:
    print('ERROR: ONNX file not found.')
else:
    try:
        _sess_opts = ort.SessionOptions()
        _sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        _sess_opts.intra_op_num_threads = os.cpu_count() or 4
        _ort_session = ort.InferenceSession(ONNX_PATH, sess_options=_sess_opts,
                                            providers=['CPUExecutionProvider'])
        _onnx_inp = _ort_session.get_inputs()[0].name
        for out in _ort_session.get_outputs():
            if out.shape and out.shape[-1] == 1536:
                _onnx_emb_key = out.name
                break
        if _onnx_emb_key is None:
            _test = np.zeros((1, CFG['perch_target']), dtype=np.float32)
            _outs = _ort_session.run(None, {_onnx_inp: _test})
            for _nm, _ov in zip([o.name for o in _ort_session.get_outputs()], _outs):
                if _ov.ndim >= 2 and _ov.shape[-1] == 1536:
                    _onnx_emb_key = _nm; break
        assert _onnx_emb_key, 'Cannot find 1536-d output.'

        _info_json = Path(ONNX_PATH).parent / 'model_info.json'
        if _info_json.exists():
            import json
            _info = json.load(open(_info_json))
            _onnx_inp     = _info.get('input_name', _onnx_inp)
            _onnx_emb_key = _info.get('embedding_output_name', _onnx_emb_key)

        _t = np.sin(2*np.pi*440*np.linspace(0,5,CFG['perch_target'])).astype(np.float32)[None]
        _o = _ort_session.run(None, {_onnx_inp: _t})
        _k = [x.name for x in _ort_session.get_outputs()].index(_onnx_emb_key)
        _e = _o[_k]
        if _e.ndim == 3: _e = _e.mean(1)
        assert _e[0].std() >= 0.05, f'ONNX std too low: {_e[0].std():.4f}'
        _onnx_ready = True
        print(f'\u2705 ONNX ready  input={_onnx_inp!r}  emb={_onnx_emb_key!r}')
        print(f'   Sanity: mean={_e[0].mean():.4f}  std={_e[0].std():.4f}')
        del _t, _o, _e
    except Exception as _ex:
        traceback.print_exc()

_out_names = [o.name for o in _ort_session.get_outputs()] if _ort_session else []
_emb_idx   = _out_names.index(_onnx_emb_key) if _onnx_emb_key in _out_names else 0
print(f'_onnx_ready: {_onnx_ready}')

✅ ONNX ready  input='inputs'  emb='embedding'
   Sanity: mean=-0.0023  std=0.1000
_onnx_ready: True


In [7]:
# === CELL 5: MEL SPECTROGRAM HELPER ===
_mel_filter = librosa.filters.mel(
    sr=CFG['mel_sr'], n_fft=CFG['n_fft'],
    n_mels=CFG['n_mels'], fmin=CFG['fmin'], fmax=CFG['fmax'],
)


def logmel_from_wave(wave_16k: np.ndarray) -> np.ndarray:
    if len(wave_16k) < CFG['mel_target']:
        wave_16k = np.pad(wave_16k, (0, CFG['mel_target'] - len(wave_16k)))
    stft   = librosa.stft(wave_16k, n_fft=CFG['n_fft'], hop_length=CFG['hop_length'],
                          window='hann', center=True)
    power  = np.abs(stft) ** 2
    mel    = _mel_filter @ power
    logmel = np.log1p(mel).astype(np.float32)
    return logmel


def mel_tensor_from_wave(wave_16k: np.ndarray) -> torch.Tensor:
    lm = logmel_from_wave(wave_16k)
    lm = (lm - lm.mean()) / (lm.std() + 1e-6)
    return torch.from_numpy(lm).float().unsqueeze(0).unsqueeze(0).to(device)


_sine = np.sin(2*np.pi*440*np.linspace(0,5,80000)).astype(np.float32)
_lm   = logmel_from_wave(_sine)
print(f'\u2705 logmel shape: {_lm.shape}')
del _sine, _lm

✅ logmel shape: (64, 251)


In [8]:
# === CELL 6: PREDICTION FUNCTION ===
_use_amp = (device.type == 'cuda')


def _perch_embs(audio_path: str, end_secs_list: list) -> dict:
    result = {}
    if _ort_session is None or not end_secs_list:
        return result
    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
        if y.ndim == 2: y = y.mean(1)
        if sr0 != CFG['perch_sr']:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr0, target_sr=CFG['perch_sr'])
        y = y.astype(np.float32)
    except Exception as _e:
        print(f'   [WARN] perch audio load failed: {_e}')
        return result

    clips = []
    for es in end_secs_list:
        end_s  = int(es * CFG['perch_sr'])
        start  = max(0, end_s - CFG['perch_target'])
        clip   = y[start:end_s]
        if len(clip) < CFG['perch_target']:
            clip = np.pad(clip, (0, CFG['perch_target'] - len(clip)))
        clips.append(clip)

    all_embs = []
    for bi in range(0, len(clips), CFG['perch_batch']):
        batch = np.stack(clips[bi:bi+CFG['perch_batch']])
        outs  = _ort_session.run(None, {_onnx_inp: batch})
        embs  = outs[_emb_idx]
        if embs.ndim == 3: embs = embs.mean(1)
        all_embs.append(embs.astype(np.float32))

    all_embs_np = np.vstack(all_embs)
    if all_embs_np.std() < 0.05:
        print(f'   [WARN] near-zero Perch embeddings for {Path(audio_path).stem}')
    for es, emb in zip(end_secs_list, all_embs_np):
        result[es] = emb
    return result


def predict_soundscape_v25(audio_path: str, end_seconds: list) -> np.ndarray:
    """
    Weighted ensemble: PerchGRU v25 (hidden=768, layers=3) + ResNet18 v25 + EfficientNet-B0 v25
    All mel models trained on correct species list.
    """
    T       = len(end_seconds)
    neutral = np.full((T, n_classes), 0.5, dtype=np.float32)

    gru_probs_list = []
    mel_probs_list = []

    # ---- Perch + GRU branch ----
    if gru_models and _onnx_ready:
        embs_map = _perch_embs(audio_path, end_seconds)
        emb_list = [
            embs_map.get(es, np.zeros(CFG['perch_emb_dim'], dtype=np.float32))
            for es in end_seconds
        ]
        emb_seq = torch.from_numpy(np.stack(emb_list)).float().unsqueeze(0).to(device)
        for m in gru_models:
            with torch.inference_mode(), autocast(enabled=_use_amp):
                probs = torch.sigmoid(m(emb_seq).float())[0].cpu().numpy()
            gru_probs_list.append(probs)
    else:
        print('   [WARN] Perch+GRU disabled')

    # ---- Mel models branch ----
    mel_models = resnet_models + effnet_models
    if mel_models and CFG['mel_weight'] > 0:
        try:
            y16, sr0 = sf.read(audio_path, always_2d=False)
            if y16.ndim == 2: y16 = y16.mean(1)
            if sr0 != CFG['mel_sr']:
                y16 = librosa.resample(y16.astype(np.float32), orig_sr=sr0, target_sr=CFG['mel_sr'])
            y16 = y16.astype(np.float32)
        except Exception as _e:
            print(f'   [WARN] mel audio load failed: {_e}')
            y16 = None

        if y16 is not None:
            mel_tens_list = []
            for es in end_seconds:
                end_samp   = int(es * CFG['mel_sr'])
                start_samp = max(0, end_samp - CFG['mel_target'])
                clip       = y16[start_samp:end_samp]
                mel_tens_list.append(mel_tensor_from_wave(clip))
            mel_batch = torch.cat(mel_tens_list, dim=0)

            for m in mel_models:
                with torch.inference_mode(), autocast(enabled=_use_amp):
                    probs = torch.sigmoid(m(mel_batch).float()).cpu().numpy()
                mel_probs_list.append(probs)

    if not gru_probs_list and not mel_probs_list:
        return neutral

    # ---- Weighted ensemble ----
    weighted_parts = []
    total_w = 0.0
    if gru_probs_list:
        weighted_parts.append(CFG['gru_weight'] * np.mean(gru_probs_list, axis=0))
        total_w += CFG['gru_weight']
    if mel_probs_list:
        weighted_parts.append(CFG['mel_weight'] * np.mean(mel_probs_list, axis=0))
        total_w += CFG['mel_weight']
    probs_mean = (np.sum(weighted_parts, axis=0) / total_w).astype(np.float32)

    # ---- Gaussian smoothing ----
    if T > 1 and CFG['gauss_sigma'] > 0:
        probs_mean = gaussian_filter1d(
            probs_mean.astype(np.float64), sigma=CFG['gauss_sigma'], axis=0
        ).astype(np.float32)

    return probs_mean


print(f'   GRU weight  : {CFG["gru_weight"]}  Mel weight: {CFG["mel_weight"]}')
print(f'   GRU models  : {len(gru_models)}')
print(f'   ResNet18    : {len(resnet_models)}')
print(f'   EfficientB0 : {len(effnet_models)}')
print('\u2705 predict_soundscape_v25() defined')

   GRU weight  : 0.9  Mel weight: 0.1
   GRU models  : 5
   ResNet18    : 5
   EfficientB0 : 5
✅ predict_soundscape_v25() defined


In [9]:
# === CELL 7: GENERATE PREDICTIONS ===
sample_sub = pd.read_csv(SAMPLE_SUB).copy()
sample_sub['_sc_id'] = sample_sub['row_id'].str.rsplit('_', n=1).str[0]
print(f'Submission rows: {len(sample_sub)}')

all_row_ids    = []
all_probs_list = []
missing_audio  = 0
error_count    = 0

for sc_id, grp in tqdm(sample_sub.groupby('_sc_id'), desc='Soundscapes', unit='file'):
    row_ids = [str(r) for r in grp['row_id']]

    audio_path = None
    for ext in ['.ogg', '.wav', '.flac']:
        c = Path(TEST_AUDIO) / f'{sc_id}{ext}'
        if c.exists():
            audio_path = str(c); break

    if audio_path is None:
        missing_audio += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    try:
        end_seconds = [int(rid.rsplit('_', 1)[-1]) for rid in row_ids]
    except Exception as _e:
        error_count += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    try:
        probs = predict_soundscape_v25(audio_path, end_seconds)
    except Exception as _e:
        print(f'WARNING: {sc_id}: {_e}')
        traceback.print_exc()
        error_count += 1
        probs = np.full((len(row_ids), n_classes), 0.5, dtype=np.float32)

    all_row_ids.extend(row_ids)
    all_probs_list.append(probs)

if missing_audio: print(f'\u26a0\ufe0f  {missing_audio} soundscapes had no audio')
if error_count:   print(f'\u26a0\ufe0f  {error_count} prediction failure(s)')
print(f'\n\u2705 Generated {len(all_row_ids)} rows')

Submission rows: 3


Soundscapes: 100%|██████████| 1/1 [00:00<00:00, 163.94file/s]

⚠️  1 soundscapes had no audio

✅ Generated 3 rows


In [10]:
# === CELL 8: BUILD & SAVE SUBMISSION ===
probs_matrix = np.concatenate(all_probs_list, axis=0)

_mean_p = probs_matrix.mean()
_std_p  = probs_matrix.std()
if abs(_mean_p - 0.5) < 0.001 and _std_p < 0.01:
    print(f'\n\u26a0\ufe0f  WARNING: all predictions near 0.5 (mean={_mean_p:.4f}, std={_std_p:.4f})')
else:
    print(f'\u2705 Distribution healthy: mean={_mean_p:.4f}, std={_std_p:.4f}')

sub_df = pd.DataFrame(probs_matrix, columns=species)
sub_df.insert(0, 'row_id', all_row_ids)
sample_cols = pd.read_csv(SAMPLE_SUB, nrows=0).columns.tolist()
sub_df = sub_df[sample_cols]

out_path = '/kaggle/working/submission.csv'
sub_df.to_csv(out_path, index=False)
print(f'\u2705 Submission saved: {out_path}  shape={sub_df.shape}')
print(sub_df.head(3))


⚠️  WARNING: all predictions near 0.5 (mean=0.5000, std=0.0000)
✅ Submission saved: /kaggle/working/submission.csv  shape=(3, 235)
                                    row_id  1161364  116570  1176823  1491113  \
0   BC2026_Test_0001_S05_20250227_010002_5      0.5     0.5      0.5      0.5   
1  BC2026_Test_0001_S05_20250227_010002_10      0.5     0.5      0.5      0.5   
2  BC2026_Test_0001_S05_20250227_010002_15      0.5     0.5      0.5      0.5   

   1595929  209233  22930  22956  22961  ...  whnjay1  whtdov  whwpic1  \
0      0.5     0.5    0.5    0.5    0.5  ...      0.5     0.5      0.5   
1      0.5     0.5    0.5    0.5    0.5  ...      0.5     0.5      0.5   
2      0.5     0.5    0.5    0.5    0.5  ...      0.5     0.5      0.5   

   y00678  yebcar  yebela1  yecmac  yecpar  yehcar1  yeofly1  
0     0.5     0.5      0.5     0.5     0.5      0.5      0.5  
1     0.5     0.5      0.5     0.5     0.5      0.5      0.5  
2     0.5     0.5      0.5     0.5     0.5      0.5      